# 02 — Enriquecimiento y Unificación

Este notebook explora los datos RAW en Snowflake, muestra la integración con Taxi Zones,
la unificación de esquemas Yellow/Green y la normalización de catálogos (vendor, payment_type, rate_code).

**Prerequisito:** Haber ejecutado `01_ingest_raw.ipynb` (datos en `NYC_TAXI_P3.RAW`).

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("NYC_Taxi_Enrichment").getOrCreate()

SF_OPTIONS = {
    "sfURL":       f"{os.environ['SF_ACCOUNT']}.snowflakecomputing.com",
    "sfUser":      os.environ["SF_USER"],
    "sfPassword":  os.environ["SF_PASSWORD"],
    "sfDatabase":  os.environ["SF_DATABASE"],
    "sfSchema":    os.environ["SF_SCHEMA_RAW"],
    "sfWarehouse": os.environ["SF_WAREHOUSE"],
    "sfRole":      os.environ["SF_ROLE"],
}

YEAR_START   = int(os.environ["YEAR_START"])
YEAR_END     = int(os.environ["YEAR_END"])
SERVICES     = os.environ["SERVICES"].split(",")

print(f"Servicios: {SERVICES}")
print(f"Rango: {YEAR_START}–{YEAR_END}")

Servicios: ['yellow', 'green']
Rango: 2024–2024


## 2.1 Exploración de datos RAW

Verificamos la estructura de `TRIPS_RAW` y conteos por servicio.

In [ ]:
print("=" * 60)
print("CELDA 2: Conteos generales por servicio y año")
print("=" * 60)

counts_sql = """
SELECT service_type, source_year, COUNT(*) AS total_rows
FROM NYC_TAXI_P3.RAW.TRIPS_RAW
GROUP BY service_type, source_year
ORDER BY service_type, source_year
"""
print(">>> Ejecutando query de conteos...")
df_counts = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", counts_sql).load())
df_counts.show(50, truncate=False)
print("✓ Conteos obtenidos")
print("=" * 60)

Py4JJavaError: An error occurred while calling o34.load.
: net.snowflake.client.jdbc.SnowflakeSQLException: JDBC driver encountered communication error. Message: HTTP status=404.
	at net.snowflake.client.core.HttpUtil.executeRequestInternal(HttpUtil.java:765)
	at net.snowflake.client.core.HttpUtil.executeRequest(HttpUtil.java:668)
	at net.snowflake.client.core.HttpUtil.executeGeneralRequest(HttpUtil.java:585)
	at net.snowflake.client.core.SessionUtil.newSession(SessionUtil.java:645)
	at net.snowflake.client.core.SessionUtil.openSession(SessionUtil.java:310)
	at net.snowflake.client.core.SFSession.open(SFSession.java:630)
	at net.snowflake.client.jdbc.DefaultSFConnectionHandler.initialize(DefaultSFConnectionHandler.java:125)
	at net.snowflake.client.jdbc.DefaultSFConnectionHandler.initializeConnection(DefaultSFConnectionHandler.java:98)
	at net.snowflake.client.jdbc.SnowflakeConnectionV1.initConnectionWithImpl(SnowflakeConnectionV1.java:142)
	at net.snowflake.client.jdbc.SnowflakeConnec

In [ ]:
print("=" * 60)
print("CELDA 3: Esquema de TRIPS_RAW (muestra)")
print("=" * 60)

schema_sql = "SELECT * FROM NYC_TAXI_P3.RAW.TRIPS_RAW LIMIT 5"
print(">>> Consultando esquema y muestra...")
df_sample = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", schema_sql).load())
df_sample.printSchema()
df_sample.show(5, truncate=False)
print("✓ Esquema mostrado")
print("=" * 60)

## 2.2 Integración con Taxi Zone Lookup

La tabla `TAXI_ZONE_LOOKUP` mapea `LocationID` a nombre de zona y borough.
Se usa para enriquecer pickup y dropoff con información geográfica legible.

In [ ]:
print("=" * 60)
print("CELDA 4: Verificar TAXI_ZONE_LOOKUP")
print("=" * 60)

zones_sql = "SELECT * FROM NYC_TAXI_P3.RAW.TAXI_ZONE_LOOKUP ORDER BY LocationID"
print(">>> Consultando zonas...")
df_zones = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", zones_sql).load())
print(f"✓ Total zonas: {df_zones.count()}")
df_zones.show(10, truncate=False)
print("=" * 60)

In [ ]:
print("=" * 60)
print("CELDA 5: Ejemplo JOIN trips + zonas")
print("=" * 60)

join_sql = """
SELECT
    r.service_type,
    r.PULocationID,
    pz.Zone    AS pu_zone,
    pz.Borough AS pu_borough,
    r.DOLocationID,
    dz.Zone    AS do_zone,
    dz.Borough AS do_borough,
    r.trip_distance,
    r.total_amount
FROM NYC_TAXI_P3.RAW.TRIPS_RAW r
LEFT JOIN NYC_TAXI_P3.RAW.TAXI_ZONE_LOOKUP pz ON r.PULocationID = pz.LocationID
LEFT JOIN NYC_TAXI_P3.RAW.TAXI_ZONE_LOOKUP dz ON r.DOLocationID = dz.LocationID
LIMIT 20
"""
print(">>> Ejecutando JOIN de enriquecimiento...")
df_join = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", join_sql).load())
df_join.show(20, truncate=False)
print("✓ JOIN completado")
print("=" * 60)

## 2.3 Unificación Yellow + Green

Yellow usa `tpep_pickup_datetime` / `tpep_dropoff_datetime`.  
Green usa `lpep_pickup_datetime` / `lpep_dropoff_datetime`.  

Se unifican con `COALESCE` para crear columnas comunes `pickup_datetime` y `dropoff_datetime`.
Ambos servicios ya están en la misma tabla `TRIPS_RAW` distinguidos por `service_type`.

In [ ]:
print("=" * 60)
print("CELDA 6: Unificación timestamps Yellow/Green")
print("=" * 60)

unify_sql = """
SELECT
    service_type,
    tpep_pickup_datetime,
    lpep_pickup_datetime,
    COALESCE(tpep_pickup_datetime, lpep_pickup_datetime)   AS pickup_datetime,
    tpep_dropoff_datetime,
    lpep_dropoff_datetime,
    COALESCE(tpep_dropoff_datetime, lpep_dropoff_datetime) AS dropoff_datetime
FROM NYC_TAXI_P3.RAW.TRIPS_RAW
WHERE tpep_pickup_datetime IS NOT NULL OR lpep_pickup_datetime IS NOT NULL
LIMIT 10
"""
print(">>> Ejecutando unificación COALESCE...")
df_unify = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", unify_sql).load())
df_unify.show(10, truncate=False)
print("✓ Unificación demostrada")
print("=" * 60)

## 2.4 Normalización de catálogos

Se decodifican los campos numéricos a nombres legibles:
- `VendorID` → `vendor_name`
- `payment_type` → `payment_type_desc`
- `RatecodeID` → `rate_code_desc`

In [ ]:
print("=" * 60)
print("CELDA 7: Decodificación de VendorID")
print("=" * 60)

vendor_sql = """
SELECT
    VendorID,
    CASE VendorID
        WHEN 1 THEN 'Creative Mobile Technologies'
        WHEN 2 THEN 'VeriFone Inc.'
        ELSE 'Unknown'
    END AS vendor_name,
    COUNT(*) AS trips
FROM NYC_TAXI_P3.RAW.TRIPS_RAW
GROUP BY VendorID
ORDER BY trips DESC
"""
print(">>> Consultando vendors...")
df_vendor = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", vendor_sql).load())
df_vendor.show(truncate=False)
print("✓ Vendors decodificados")
print("=" * 60)

In [ ]:
print("=" * 60)
print("CELDA 8: Decodificación de payment_type")
print("=" * 60)

payment_sql = """
SELECT
    payment_type,
    CASE payment_type
        WHEN 1 THEN 'Credit card'
        WHEN 2 THEN 'Cash'
        WHEN 3 THEN 'No charge'
        WHEN 4 THEN 'Dispute'
        WHEN 5 THEN 'Unknown'
        WHEN 6 THEN 'Voided trip'
        ELSE 'Other'
    END AS payment_type_desc,
    COUNT(*) AS trips
FROM NYC_TAXI_P3.RAW.TRIPS_RAW
GROUP BY payment_type
ORDER BY trips DESC
"""
print(">>> Consultando payment types...")
df_pay = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", payment_sql).load())
df_pay.show(truncate=False)
print("✓ Payment types decodificados")
print("=" * 60)

In [ ]:
print("=" * 60)
print("CELDA 9: Decodificación de RatecodeID")
print("=" * 60)

rate_sql = """
SELECT
    RatecodeID::INTEGER AS rate_code_id,
    CASE RatecodeID::INTEGER
        WHEN 1 THEN 'Standard rate'
        WHEN 2 THEN 'JFK'
        WHEN 3 THEN 'Newark'
        WHEN 4 THEN 'Nassau/Westchester'
        WHEN 5 THEN 'Negotiated fare'
        WHEN 6 THEN 'Group ride'
        ELSE 'Unknown'
    END AS rate_code_desc,
    COUNT(*) AS trips
FROM NYC_TAXI_P3.RAW.TRIPS_RAW
GROUP BY RatecodeID::INTEGER
ORDER BY trips DESC
"""
print(">>> Consultando rate codes...")
df_rate = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", rate_sql).load())
df_rate.show(truncate=False)
print("✓ Rate codes decodificados")
print("=" * 60)

## 2.5 Auditoría de ingesta

Revisamos `INGESTION_LOG` para confirmar el estado de cada mes/servicio cargado.

In [ ]:
print("=" * 60)
print("CELDA 10: Auditoría — INGESTION_LOG")
print("=" * 60)

log_sql = """
SELECT run_id, service_type, source_year, source_month,
       rows_loaded, status, started_at_utc, finished_at_utc
FROM NYC_TAXI_P3.RAW.INGESTION_LOG
ORDER BY service_type, source_year, source_month
"""
print(">>> Consultando log de auditoría...")
df_log = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", log_sql).load())
print(f"✓ Registros de auditoría: {df_log.count()}")
df_log.show(50, truncate=False)
print("✓ NB02 completado — Enriquecimiento y Unificación")
print("=" * 60)

Py4JJavaError: An error occurred while calling o45.load.
: net.snowflake.client.jdbc.SnowflakeSQLException: JDBC driver encountered communication error. Message: HTTP status=404.
	at net.snowflake.client.core.HttpUtil.executeRequestInternal(HttpUtil.java:765)
	at net.snowflake.client.core.HttpUtil.executeRequest(HttpUtil.java:668)
	at net.snowflake.client.core.HttpUtil.executeGeneralRequest(HttpUtil.java:585)
	at net.snowflake.client.core.SessionUtil.newSession(SessionUtil.java:645)
	at net.snowflake.client.core.SessionUtil.openSession(SessionUtil.java:310)
	at net.snowflake.client.core.SFSession.open(SFSession.java:630)
	at net.snowflake.client.jdbc.DefaultSFConnectionHandler.initialize(DefaultSFConnectionHandler.java:125)
	at net.snowflake.client.jdbc.DefaultSFConnectionHandler.initializeConnection(DefaultSFConnectionHandler.java:98)
	at net.snowflake.client.jdbc.SnowflakeConnectionV1.initConnectionWithImpl(SnowflakeConnectionV1.java:142)
	at net.snowflake.client.jdbc.SnowflakeConnec

## Resumen

- **Taxi Zone Lookup** integrada: permite JOINs por `PULocationID` / `DOLocationID` → zona, borough.
- **Unificación Yellow/Green**: `COALESCE(tpep_*, lpep_*)` crea timestamps unificados.
- **Catálogos normalizados**: VendorID, payment_type, RatecodeID decodificados a nombres legibles.
- Todo listo para construir la OBT en `03_construccion_obt.ipynb`.